In [5]:
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os
import json
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

In [22]:
# import requests
# import zipfile
# from pathlib import Path
# data_path = Path("data/")
# image_path=data_path/"images"
# image_path.mkdir(parents=True,exist_ok=True)

In [23]:
# with zipfile.ZipFile("converted.zip", 'r') as zip_ref:
#   zip_ref.extractall(image_path)

In [52]:
class RotatedWordDataset(Dataset):
    def __init__(self, bangla_image_dir, bangla_annotation_dir, english_image_dir, img_size, transform=None):
        self.bangla_image_dir = bangla_image_dir
        self.bangla_annotation_dir = bangla_annotation_dir
        self.english_image_dir = english_image_dir
        self.transform = transform
        self.img_size = img_size
        self.samples = []  # List of (PIL.Image, rotation_label)

        self._prepare_dataset()

    def _prepare_dataset(self):
        for filename in os.listdir(self.bangla_annotation_dir):
            if not filename.endswith(".json"):
                continue
            image_name = filename.replace(".json", ".jpg")
            image_path = os.path.join(self.bangla_image_dir, image_name)
            json_path = os.path.join(self.bangla_annotation_dir, filename)

            try:
                img = Image.open(image_path).convert("RGB")
            except:
                continue

            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            for shape in data.get("shapes", []):
                points = shape["points"]
                (x1, y1), (x2, y2) = points
                try:
                    if(x2>x1 and y2>y1):
                        crop = img.crop((x1, y1, x2, y2)).resize(self.img_size)
                except Exception as e:
                    print(f"Error cropping: label={shape['label']} box=({x1}, {y1}, {x2}, {y2}) -> {e}")

                # print (x1, x2, y1, y2)
                # crop = img.crop((x1, y1, x2, y2)).resize(self.img_size)

                self.samples.append((crop, 0))
                self.samples.append((crop.rotate(-90, expand=True).resize(self.img_size), 1))
                self.samples.append((crop.rotate(-180, expand=True).resize(self.img_size), 2))
                self.samples.append((crop.rotate(-270, expand=True).resize(self.img_size), 3))

        for subdir, _, files in os.walk(self.english_image_dir):
            for file in files:
                if file.endswith('.png'):
                    file_path = os.path.join(subdir, file)
                    try:
                        e_img = Image.open(file_path)
                        # if e_img.size[0] < 35 or e_img.size[1] < 35:
                        #     # print(f"Skipping small image: {file_path}, size={e_img.size}")
                        #     continue
        
                        e_img = e_img.resize(self.img_size)
                        self.samples.append((e_img, 0))
                        self.samples.append((e_img.rotate(-90, expand=True).resize(self.img_size), 1))
                        self.samples.append((e_img.rotate(-180, expand=True).resize(self.img_size), 2))
                        self.samples.append((e_img.rotate(-270, expand=True).resize(self.img_size), 3))
                    except Exception as e:
                        print(f"Skipping unreadable image {file_path}: {e}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image, label = self.samples[idx]
        image
        # Convert to grayscale tensor
        image = TF.to_grayscale(image, num_output_channels=1)
        image = TF.to_tensor(image)  # Returns tensor [1, H, W] in range [0, 1]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label)

In [53]:
dataset = RotatedWordDataset(
    bangla_image_dir="/kaggle/input/bangla-word/raw",
    bangla_annotation_dir="/kaggle/input/bangla-word/raw",
    english_image_dir="/kaggle/input/english-word-simplified",
    img_size=(32, 32)
)

# img=Image.open("/kaggle/input/english-word-simplified/a01-003u/a01-003u-00-00.png")
# plt.imshow(img)
# img=img.resize( (64, 64))
# plt.imshow(img)
# plt.show()
#dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
# image, label= dataset[112]
# dataset[1112]
# image.shape
# image = TF.to_pil_image(image)

# plt.imshow(image)
# plt.title(f"Label: {label}")

Skipping unreadable image /kaggle/input/english-word-simplified/a01-117/a01-117-05-02.png: cannot identify image file '/kaggle/input/english-word-simplified/a01-117/a01-117-05-02.png'


In [26]:
type(dataset[1])

tuple

In [54]:
from torch.utils.data import random_split
train_size=int( 0.8 * len(dataset.samples))
test_size =int (len(dataset.samples) - train_size)
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
# image,label= train_dataset[133]
# plt.imshow(image)
# plt.title(f"Label: {label}")
# type(train_dataset)

In [55]:
len(dataset.samples)


193348

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True)
image_batch, label_batch= next(iter(train_dataloader))
image_batch.shape, label_batch
for img, label in zip(image_batch, label_batch):
    img_np = img.squeeze().numpy()  # Remove channel dim -> shape becomes (64, 64)
    img.shape
    plt.title(f"Label: {label.item()}")
    plt.imshow(img_np, cmap='gray')
    plt.axis('off')
    plt.show()

In [30]:
len(train_dataloader)

4447

In [6]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor, Lambda, Compose
import matplotlib.pyplot as plt

class rotated_word_model_0(nn.Module):
  def __init__(self, input_shape, hidden_units, output_shape):
    super().__init__()
    self.conv_block_1= nn.Sequential(
      nn.Conv2d(in_channels=input_shape, out_channels=hidden_units,
                kernel_size=3, stride=1, padding=1),
      nn.ReLU(),
      nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units,
                kernel_size=3, stride=1, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),
    )
    self.conv_block_2= nn.Sequential(
      nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units,
                kernel_size=3, stride=1, padding=1),
      nn.ReLU(),
      nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units,
                kernel_size=3, stride=1, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),
    )
    self.conv_block_3= nn.Sequential(
      nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units,
                kernel_size=3, stride=1, padding=1),
      nn.ReLU(),
      nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units,
                kernel_size=3, stride=1, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),
    )
    self.conv_block_4= nn.Sequential(
      nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units,
                kernel_size=3, stride=1, padding=1),
      nn.ReLU(),
      nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units,
                kernel_size=3, stride=1, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),
    )  
    self.classifier= nn.Sequential(
      nn.Flatten(),
      nn.Linear(in_features=int(hidden_units*16*(1/4)), out_features=output_shape)
    )
  def forward(self, x):
    x= self.conv_block_1(x)
    #print(x.shape)
    x= self.conv_block_2(x)
    x= self.conv_block_3(x)
    x= self.conv_block_4(x)
    #print(x.shape)
    x= self.classifier(x)
    return x



In [32]:
model_0= rotated_word_model_0(input_shape=1, hidden_units=40, output_shape=4)
#list(mdl0.parameters())
#model_0(image_batch)

In [33]:
try:
  import torchinfo
except:
  !pip install torchinfo
  import torchinfo
from torchinfo import summary
#summary(model_0, input_size=(32, 3, 128, 128))

In [34]:
# Calculate accuracy (a classification metric)
# def accuracy_fn(y_true, y_pred):
#     correct = torch.eq(y_true, y_pred).sum().item() # torch.eq() calculates where two tensors are equal
#     acc = (correct / len(y_pred)) * 100
#     return acc

In [35]:
  def test_step(model_0: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn
              ):
    test_loss, test_acc = 0, 0
    model_0.eval()
    with torch.inference_mode():
      for batch, (X, y) in enumerate(dataloader):
        # 1. Forward pass
        test_pred = model_0(X)
        #print("TEST FRWRD")
        # 2. Calculate loss (accumulatively)
        test_loss += loss_fn(test_pred, y) # accumulatively add up the loss per epoch
        #print("TEST LOSS")
        # 3. Calculate accuracy (preds need to be same as y_true)
        test_pred_labels = torch.argmax(torch.softmax(test_pred, dim=1), dim=1)
        test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))

      # Calculations on test metrics need to happen inside torch.inference_mode()
      # Divide total test loss by length of test dataloader (per batch)
      test_loss /= len(test_dataloader)

      # Divide total accuracy by length of test dataloader (per batch)
      test_acc /= len(test_dataloader)
      return test_loss, test_acc

    ## Print out what's happening
    print(f"\nTrain loss: {train_loss:.5f} | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%\n")



In [36]:
# Import tqdm for progress bar
from tqdm.auto import tqdm
import torch.optim as optim
optimizer = torch.optim.SGD(params=model_0.parameters(), lr=0.1)
# Set the seed and start the timer
torch.manual_seed(42)
#train_time_start_on_cpu = timer()
loss_fn = nn.CrossEntropyLoss()

# Set the number of epochs (we'll keep this small for faster training times)

def train_step(model_0: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
              ):
  model_0.train()

  # Create training and testing loop

  ### Training
  train_loss, train_acc = 0, 0
  # Add a loop to loop through training batches
  for batch, (X, y) in enumerate(dataloader):

    #print(f"batch: {batch}")
    # 1. Forward pass
    y_pred = model_0(X)
    #print(f"Forward pass")

    # 2. Calculate loss (per batch)
    loss = loss_fn(y_pred, y)
    train_loss += loss # accumulatively add up the loss per epoch
    #print(f"Calculate loss (per batch) {loss}")

    # 3. Optimizer zero grad
    optimizer.zero_grad()
    #print("Optimizer zero grad")

    # 4. Loss backward
    loss.backward()
    #print("Loss backward")

    # 5. Optimizer step
    optimizer.step()

    y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
    train_acc += (y_pred_class == y).sum().item()/len(y_pred)
    #print("Optimizer step")
    # Print out how many samples have been seen
    if batch % 400 == 0:
      print(f"Looked at {batch * len(X)}/{len(train_dataloader.dataset)} samples")

  # Divide total train loss by length of train dataloader (average loss per batch per epoch)
  train_loss /= len(train_dataloader)
  train_acc /= len(train_dataloader)
  return train_loss, train_acc



In [37]:
def train(model_0:torch.nn.Module,
          train_dataloader,
          test_dataloader,
          optimizer,
          loss_fn,
          epochs:int=5):
  results ={
      "train_loss":[],
      "train_acc":[],
      "test_loss":[],
      "test_acc":[]
  }
  best_acc=0.0
  for epoch in tqdm(range(epochs)):
    train_loss, train_acc = train_step(model_0=model_0,
                                       dataloader=train_dataloader,
                                       loss_fn=loss_fn,
                                       optimizer=optimizer)
    
        
    test_loss, test_acc =test_step(model_0=model_0,
                                    dataloader=test_dataloader,
                                    loss_fn=loss_fn)
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model_0.state_dict(), "best_model_1.pth")
    print(f"Epoch: {epoch} | Train loss: {train_loss:.5f} | Train acc: {train_acc:.2f}% | Test loss: {test_loss:.5f} | Test acc: {test_acc:.2f}%")
    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)
  return results


In [57]:
torch.manual_seed(42)
EPOCHS= 5
from timeit import default_timer as timer
start_time = timer()
model_0_results= train(model_0=model_0,
          train_dataloader=train_dataloader,
          test_dataloader=test_dataloader,
          optimizer=optimizer,
          loss_fn=loss_fn,
          epochs=EPOCHS)
end_time = timer()
print(f"Total training time: {end_time-start_time:.3f} seconds")

  0%|          | 0/5 [00:00<?, ?it/s]

Looked at 0/154678 samples
Looked at 12800/154678 samples
Looked at 25600/154678 samples
Looked at 38400/154678 samples
Looked at 51200/154678 samples
Looked at 64000/154678 samples
Looked at 76800/154678 samples
Looked at 89600/154678 samples
Looked at 102400/154678 samples
Looked at 115200/154678 samples
Looked at 128000/154678 samples
Looked at 140800/154678 samples
Looked at 153600/154678 samples
Epoch: 0 | Train loss: 0.25769 | Train acc: 0.89% | Test loss: 0.24991 | Test acc: 0.90%
Looked at 0/154678 samples
Looked at 12800/154678 samples
Looked at 25600/154678 samples
Looked at 38400/154678 samples
Looked at 51200/154678 samples
Looked at 64000/154678 samples
Looked at 76800/154678 samples
Looked at 89600/154678 samples
Looked at 102400/154678 samples
Looked at 115200/154678 samples
Looked at 128000/154678 samples
Looked at 140800/154678 samples
Looked at 153600/154678 samples
Epoch: 1 | Train loss: 0.23091 | Train acc: 0.90% | Test loss: 0.22229 | Test acc: 0.91%
Looked at 0/15

In [58]:
torch.save(model_0.state_dict(), "best_combined_model_1.pth")

In [ ]:
torch.save(model_0.state_dict(), "best_model.pth")

In [7]:
import torch
new_model=rotated_word_model_0(input_shape=1, hidden_units=40, output_shape=4)
new_model.load_state_dict(torch.load("/kaggle/input/default/pytorch/default/1/best_combined_model_1.pth", weights_only=True))


<All keys matched successfully>

In [8]:
from typing import List
import torchvision
import PIL

def pred_and_plot_image(model:torch.nn.Module ,
                       image:PIL.Image.Image,
                       class_names: List[str]= None,
                       transform=None):
    #target_image= TF.to_grayscale(target_image, num_output_channels=1),
    #target_image= torchvision.io.read_image(str(image_path)).type(torch.float32)
    image = image.convert("L")
    target_image= TF.to_tensor(image).type(torch.float32)
    #target_image= target_image/255
    if transform:
        target_image= transform(target_image)
        temp= target_image
    model.eval()
    with torch.inference_mode():
        target_image= target_image.unsqueeze(0)
        target_image_pred= model(target_image)

    target_image_pred_probs= torch.softmax(target_image_pred, dim=1)
    target_image_pred_label= torch.argmax(target_image_pred_probs, dim=1)
    #plt.imshow(TF.to_pil_image(temp))
    if class_names:
        title= class_names[target_image_pred_label]
        #f"Pred: {class_names[target_image_pred_label]} | prob: {target_image_pred_probs.max()}"
    else:
        title= target_image_pred_label
        #f"Pred: {target_image_pred_label} | Prob: {target_image_pred_probs.max()}"
    # plt.title(title)
    # plt.axis(False)
    return title #title is the prediction

In [7]:
from torchvision import transforms
custom_image_transform= transforms.Compose([
    transforms.Resize(size=(32, 32))
])
#custom_image_path="/kaggle/input/tttest2/2025-05-15 20_47_18-DSD note (Arafat).pdf - Adobe Acrobat Reader (64-bit).png"
class_names=[0,1,2,3]

In [ ]:
# pred_and_plot_image(model= model_0,
#                    image_path=custom_image_path,
#                    class_names= class_names,
#                    transform=custom_image_transform,
#                    )

In [9]:
!pip install easyocr
!pip install pymupdf
import easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.5 MB/s eta 0:00:000:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 35.7 MB/s eta 0:00:0000:0100:01m
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstalling nvidia-nvjitlink-cu12-12.8.93:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.8.93
  Attempting uninstall: nvidia-curand-cu12
    Found existing installation: nvidia-curand-cu12 10.3.9.90
    Uninstalling nvidia-curand-cu12-1

In [24]:

from PIL import Image
import fitz  # PyMuPDF
import matplotlib.pyplot as plt
import numpy as np
import io
def cropp_images_from_pdf (pdf_file):
    reader = easyocr.Reader(['en','bn'])
    # pdf_path= "/kaggle/input/test-pdf-2/2025-05-17 01_13_43-DSD note (Arafat).pdf - Adobe Acrobat Reader (64-bit).pdf"
    doc = fitz.open(pdf_file)
    #pdf_bytes = pdf_file.read()
    #doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    page_images = [] 
    
    all_pages_data = [] # save page numbers and coordinate of words
    real_page_images=[]
    for page_number, page in enumerate(doc, start=1):
        
        print(f"working on page {page_number}")
        pix = page.get_pixmap(dpi=400)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=95, dpi=(400, 400))
        buf.seek(0)  # Rewind buffer
        real_page_images.append(buf)
        
        img = img.convert("L")
        page_images.append(img)
        
        img_np = np.array(img)
        
        #reading one page
        results= reader.readtext(img_np) 
        
        page_info={
            "page": page_number,
            "words":[]
        }
        for bbox, text, confidence in results:
            word_data = {
                #"text": text,
                #"confidence": confidence,
                "box": bbox  # list of 4 points
            }
            page_info["words"].append(word_data)
            
        all_pages_data.append(page_info)
    all_pages_with_cropped_word=[]
    
    for i, page_info in enumerate(all_pages_data):
        crop_info={
            "page": page_info["page"],
            "cropped_images": []
        }
        print(f"cropping from page {crop_info['page']} ")
        
        for word_data in page_info["words"]:
            box = word_data["box"]
            x_coords = [p[0] for p in box]
            y_coords = [p[1] for p in box]
            x_min, x_max = int(min(x_coords)), int(max(x_coords))
            y_min, y_max = int(min(y_coords)), int(max(y_coords))
            
            cropped_image = page_images[i].crop((x_min, y_min, x_max, y_max))
            crop_info["cropped_images"].append(cropped_image)
    
        all_pages_with_cropped_word.append(crop_info)

    return all_pages_with_cropped_word, real_page_images

In [26]:
from torchvision import transforms
def rotation_prediction_of_cropped_images(all_pages_with_cropped_word):
    cropped_rotation_info=[]
    class_names=[0,1,2,3]
    custom_image_transform= transforms.Compose([
        transforms.Resize(size=(32, 32))
    ])
    for i, crop_info in enumerate(all_pages_with_cropped_word):
        cropped_images= crop_info["cropped_images"]
        for cropped_image in cropped_images:
            
            rotation=pred_and_plot_image(model= new_model,
                       image=cropped_image,
                       class_names= class_names,
                       transform=custom_image_transform,
                       )
            rotation_info={
                "page":crop_info["page"],
                "rotation":rotation
            }
            cropped_rotation_info.append(rotation_info)

    return cropped_rotation_info
#         plt.imshow(cropped_image)
#         plt.title(rotation)
#         plt.axis('off')
#         plt.show()
# for i, rotation_info in enumerate(cropped_rotation_info):
#     print(f"page: {rotation_info['page']} | rotation: {rotation_info['rotation'] }")

In [14]:
from collections import Counter
def group_rotations_by_page(cropped_rotation_info):
    # Create a dictionary to store rotations grouped by page
    rotations_by_page = {}
    
    for info in cropped_rotation_info:
        page = info["page"]
        rotation = info["rotation"]
        
        # If page is not in dictionary, add it with an empty list
        if page not in rotations_by_page:
            rotations_by_page[page] = []
        
        # Append the rotation to the corresponding page
        rotations_by_page[page].append(rotation)
    
    return rotations_by_page


def get_most_common_rotation_by_page(cropped_rotation_info):

    rotations_by_page = group_rotations_by_page(cropped_rotation_info)
    
    # Create a result dictionary to store the most common rotation for each page
    most_common_rotations = []
    
    # Find the most common rotation for each page
    for page, rotations in rotations_by_page.items():
        counter = Counter(rotations)
        most_common = counter.most_common(1)[0][0]  # Get the most common rotation value
        most_common_rotations.append(most_common)
    
    return most_common_rotations
    #, rotations_by_page
    
#most_common_rotations= get_most_common_rotation_by_page(cropped_rotation_info)

# for page in range( len(most_common_rotations)):
#     print(f" page: {page} | common rotation: {most_common_rotations[page]}")

In [28]:
import cv2
import numpy as np
from IPython.display import FileLink
def final_rotated_pdf(most_common_rotations, real_page_images):
    for page in range(len(most_common_rotations)):
        print(f"page {page}")
        img = real_page_images[page]  # This is either a BytesIO (initially) or a PIL.Image (after first rotation)
    
        # Only open if it's a BytesIO
        if isinstance(img, io.BytesIO):
            img = Image.open(img)
        
        img_np = np.array(img)
        k = most_common_rotations[page]  # 0, 1, 2, or 3
        rotated_np = np.rot90(img_np, k=k)
        rotated_img = Image.fromarray(rotated_np)
        print(f"Size after saving: width = {rotated_img.width}, height = {rotated_img.height}")
        if most_common_rotations[page] == 1 or most_common_rotations[page] == 3 :
            rotated_img= rotated_img.resize((rotated_img.height, rotated_img.width))
        real_page_images[page] = rotated_img
        
      
    pdf_images = []
    for img in real_page_images:
        if img.mode != 'RGB':
            img = img.convert('RGB')
        pdf_images.append(img)
    
    # Save all pages into a single PDF
    output_pdf_path = "rotated_output.pdf"
    pdf_images[0].save(
        output_pdf_path,
        save_all=True,
        append_images=pdf_images[1:]
    )

    return output_pdf_path
    # FileLink("rotated_output.pdf")

# for img in real_page_images:
#     plt.imshow(img)
#     plt.show()
# len(real_page_images)

In [31]:
def prediction_and_final_pdf(pdf_file ):
    all_pages_with_cropped_word, real_page_images = cropp_images_from_pdf(pdf_file)
    cropped_rotation_info = rotation_prediction_of_cropped_images(all_pages_with_cropped_word)
    most_common_rotations = get_most_common_rotation_by_page(cropped_rotation_info)
    output_pdf_path = final_rotated_pdf(most_common_rotations, real_page_images)
    return output_pdf_path

In [ ]:
!pip install gradio

In [32]:
import gradio as gr


def greet(pdf_file):
    return pdf_file
demo = gr.Interface(
    fn=prediction_and_final_pdf,
    inputs=gr.File(file_types=[".pdf"], type="filepath", label="Upload a PDF"),
    outputs=gr.File(label="Returned PDF")
)

demo.launch()


* Running on local URL:  http://127.0.0.1:7868
It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://86e197f91cfb63fd67.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


working on page 1
cropping from page 1 
page 0
Size after saving: width = 3306, height = 4678
